# M7.6 · Export ve deployment

Modeli Python'dan çıkarıp çalıştırılabilir bir grafa dönüştürmek. Boyut ve verim hesaplarını ve tracing'in kontrol akışını nasıl "dondurduğunu" elle görüyor, saf Python'da modeli JSON'a yazıp yorumlayan bir mini çalışma zamanı ve kayıt tabanlı bir izleyici yazıyor, NumPy'da ONNX benzeri düğüm listesini çalıştıran, sabit katlayan ve sembolik batch ile şekil çıkaran bir çalışma zamanı kuruyor, `torch.export` ile dinamik şekilli ve `torch.cond`'lu modeller dışa aktarıyor ve en sonda AOTInductor ile modeli Python'suz çalışabilen derlenmiş bir pakete dönüştürüyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m7-6-export.html](https://m-gonul.github.io/pytorch-lab/m7-6-export.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m7_6_export.py
CHECKS = {
 "b": "import json\nimport random\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n    rng = random.Random(0)\n    layers = [{\"op\": \"linear\", \"W\": [[rng.uniform(-1, 1) for _ in range(3)] for _ in range(4)], \"b\": [0.1, -0.2, 0.0, 0.3]},\n              {\"op\": \"relu\"},\n              {\"op\": \"linear\", \"W\": [[rng.uniform(-1, 1) for _ in range(4)] for _ in range(2)], \"b\": [0.0, 1.0]}]\n\n    def js():\n        s = need(ns, \"to_json\")(layers)\n        ensure(isinstance(s, str) and json.loads(s) == layers, \"geçerli JSON\")\n        x = [0.5, -1.0, 2.0]\n        close(need(ns, \"run\")(need(ns, \"from_json\")(s), x), ns[\"run\"](layers, x))\n        h = [max(0.0, sum(w * v for w, v in zip(r, x)) + b) for r, b in zip(layers[0][\"W\"], layers[0][\"b\"])]\n        close(ns[\"run\"](layers, x), [sum(w * v for w, v in zip(r, h)) + b for r, b in zip(layers[2][\"W\"], layers[2][\"b\"])])\n    R.test(\"JSON gidiş-dönüş ve yorumlayıcı\", js)\n\n    def tr():\n        f = lambda x: x * 2 if x.sum() > 0 else -x\n        g = need(ns, \"trace\")(f, [1.0, 2.0])\n        ensure([tuple(o) for o in g] == [(\"mul\", 2)], f\"graf {g}\")\n        out = need(ns, \"replay\")(g, [-1.0, -1.0])\n        close(out, [-2.0, -2.0], what=\"donmuş dal\")\n        g2 = ns[\"trace\"](f, [-1.0, -1.0])\n        close(ns[\"replay\"](g2, [-1.0, -1.0]), [1.0, 1.0], what=\"negatif örnekle izlenince öbür dal\")\n        return \"izleme girdinin aldığı dalı donduruyor: x = (−1, −1) için 2x hesaplandı\"\n    R.test(\"tracing kontrol akışını donduruyor\", tr)\n\n    def chain():\n        g = need(ns, \"trace\")(lambda x: -(3 * (x * 0.5)), [1.0])\n        close(need(ns, \"replay\")(g, [4.0, -2.0]), [-6.0, 3.0])\n    R.test(\"zincirlenmiş işlemler (sağdan çarpma dahil)\", chain)\n",
 "c": "import numpy as np\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    rng = np.random.default_rng(0)\n    inits = {\"W1\": rng.normal(size=(8, 4)), \"b1\": rng.normal(size=8), \"two\": np.array(2.0), \"W2\": rng.normal(size=(8, 3)), \"b2\": rng.normal(size=3)}\n    nodes = [\n        {\"op\": \"Gemm\", \"inputs\": [\"x\", \"W1\", \"b1\"], \"output\": \"h\"},\n        {\"op\": \"Relu\", \"inputs\": [\"h\"], \"output\": \"r\"},\n        {\"op\": \"Mul\", \"inputs\": [\"W2\", \"two\"], \"output\": \"W2s\"},\n        {\"op\": \"Transpose\", \"inputs\": [\"W2s\"], \"output\": \"W2t\"},\n        {\"op\": \"Gemm\", \"inputs\": [\"r\", \"W2t\", \"b2\"], \"output\": \"y\"},\n        {\"op\": \"Softmax\", \"inputs\": [\"y\"], \"output\": \"p\"},\n    ]\n    x = rng.normal(size=(5, 4))\n    ref = np.maximum(x @ inits[\"W1\"].T + inits[\"b1\"], 0) @ (2 * inits[\"W2\"]) + inits[\"b2\"]\n    ref = np.exp(ref - ref.max(1, keepdims=True)); ref /= ref.sum(1, keepdims=True)\n\n    R.test(\"run_graph\", lambda: close(need(ns, \"run_graph\")(nodes, inits, {\"x\": x})[\"p\"], ref))\n\n    def fold():\n        n2, i2 = need(ns, \"fold_constants\")(nodes, inits)\n        ensure([n[\"op\"] for n in n2] == [\"Gemm\", \"Relu\", \"Gemm\", \"Softmax\"], f\"katlama sonrası {[n['op'] for n in n2]}\")\n        ensure(\"W2t\" in i2 and \"W2t\" not in inits, \"katlanan sabit yeni inits'te; orijinal sözlük değişmemeli\")\n        close(ns[\"run_graph\"](n2, i2, {\"x\": x})[\"p\"], ref, what=\"katlanmış graf aynı sonuç\")\n    R.test(\"fold_constants\", fold)\n\n    def shapes():\n        sh = need(ns, \"infer_shapes\")(nodes, inits, {\"x\": (\"N\", 4)})\n        want = {\"h\": (\"N\", 8), \"r\": (\"N\", 8), \"W2s\": (8, 3), \"W2t\": (3, 8), \"y\": (\"N\", 3), \"p\": (\"N\", 3)}\n        for k, v in want.items():\n            ensure(tuple(sh[k]) == v, f\"{k}: {sh[k]}, beklenen {v}\")\n        sh2 = ns[\"infer_shapes\"]([{\"op\": \"Add\", \"inputs\": [\"a\", \"b\"], \"output\": \"c\"}], {}, {\"a\": (\"N\", 1, 5), \"b\": (7, 1)})\n        ensure(tuple(sh2[\"c\"]) == (\"N\", 7, 5), f\"yayınlama: {sh2['c']}\")\n    R.test(\"infer_shapes (sembolik batch, yayınlama)\", shapes)\n",
 "d": "import logging\nimport torch\nfrom torch.export import export\nfrom labcheck import close, need, ensure\n\ntorch._logging.set_logs(dynamo=logging.ERROR, export=logging.ERROR)\n\n\ndef check(ns, R):\n    torch.manual_seed(0)\n    m = need(ns, \"MLP\", \"sınıf\")().eval()\n\n    def dyn():\n        ep = need(ns, \"export_dynamic\")(m, torch.randn(4, 16))\n        for B in (1, 7, 300):\n            x = torch.randn(B, 16)\n            close(ep.module()(x), m(x), what=f\"batch {B}\")\n        names = need(ns, \"op_names\")(ep)\n        ensure(sum(\"linear\" in n for n in names) == 2 and any(\"relu\" in n for n in names), f\"işlemler {names}\")\n        return \"ATen işlemleri: \" + \", \".join(names)\n    R.test(\"dinamik batch ile export\", dyn)\n\n    def sl():\n        ep = ns[\"export_dynamic\"](m, torch.randn(4, 16))\n        ep2 = need(ns, \"save_load\")(ep)\n        x = torch.randn(5, 16)\n        close(ep2.module()(x), m(x))\n    R.test(\"save/load (.pt2) gidiş-dönüş\", sl)\n\n    def bad():\n        try:\n            export(need(ns, \"BadModel\", \"sınıf\")(), (torch.randn(3),))\n        except Exception as e:\n            return f\"Python if → {type(e).__name__}\"\n        raise AssertionError(\"BadModel export'u hata vermeliydi\")\n    R.test(\"veriye bağlı if export'ta reddediliyor\", bad)\n\n    def cond():\n        ep = export(need(ns, \"CondModel\", \"sınıf\")(), (torch.randn(3),))\n        for v in (torch.tensor([1.0, 2.0, 3.0]), torch.tensor([-1.0, -1.0, 0.5])):\n            want = v * 2 if v.sum() > 0 else -v\n            close(ep.module()(v), want, what=f\"girdi {v.tolist()}\")\n        ensure(any(\"cond\" in str(n.target) for n in ep.graph.nodes if n.op == \"call_function\"), \"grafta cond düğümü olmalı\")\n    R.test(\"torch.cond: iki dal da grafta\", cond)\n",
 "e": "import os\nimport torch\nfrom labcheck import close, need, ensure, bench, fmt_time\n\n\ndef check(ns, R):\n    m = need(ns, \"m\", \"model\")\n    run = need(ns, \"run\", \"yüklenmiş paket\")\n    path = need(ns, \"path\", \"paket yolu\")\n\n    def eq():\n        ensure(os.path.exists(path), \"paket dosyası yok\")\n        with torch.inference_mode():\n            for B in (1, 9, 128):\n                x = torch.randn(B, 64)\n                close(run(x), m(x), atol=1e-5, what=f\"batch {B}\")\n        return f\"paket: {os.path.getsize(path):,} bayt\"\n    R.test(\"AOTInductor paketi = eager (farklı batch'ler)\", eq)\n\n    def reload():\n        r2 = need(ns, \"load_package\")(path)\n        x = torch.randn(3, 64)\n        with torch.inference_mode():\n            close(r2(x), m(x), atol=1e-5)\n    R.test(\"paket yeniden yüklenebiliyor\", reload)\n    x1, x128 = torch.randn(1, 64), torch.randn(128, 64)\n    with torch.inference_mode():\n        rows = []\n        for name, fn in ((\"eager\", m), (\"AOTInductor\", run)):\n            rows.append([name, fmt_time(bench(lambda: fn(x1), repeat=5)), fmt_time(bench(lambda: fn(x128), repeat=5))])\n    R.table(\"Çıkarım gecikmesi (CPU)\", [\"\", \"batch 1\", \"batch 128\"], rows,\n            note=\"Küçük batch'te Python ve modül çağrı yükü baskındır; derlenmiş paket bunu ve ara tensörleri ortadan kaldırır.\")\n"
}

In [ ]:
import importlib, labcheck, checks_m7_6_export as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

Eğitimde model Python'da yaşar; dağıtımda genelde **Python'suz**, sabit bir graf olarak çalıştırılmak istenir.

**Grafı yakalamanın iki yolu:**

- *Tracing* (eski `torch.jit.trace`, ONNX'in eski yolu): örnek bir girdiyle çalıştır, yapılan işlemleri kaydet. Kontrol akışı **o girdinin aldığı dalda donar**.
- *`torch.export`* (Dynamo tabanlı): bytecode'u sembolik olarak izler; veriye bağlı kontrol akışında hata verir, şekillere bağlı koşullar için **guard** ekler. Dinamik boyutlar `torch.export.Dim` ile işaretlenir; veriye bağlı dallar `torch.cond` ile ifade edilir.

Çıktı bir `ExportedProgram`: ATen düzeyinde işlemlerden oluşan bir FX grafı + ağırlıklar. Buradan:

- `torch.export.save` → `.pt2` dosyası,
- `torch.onnx.export(…, dynamo=True)` → ONNX (ONNX Runtime, tarayıcıda ONNX Runtime Web),
- `torch._inductor.aoti_compile_and_package` → C++'tan yüklenebilen derlenmiş paket (AOTInductor).

Çalışma zamanları grafı tek başına optimize eder: **sabit katlama** (yalnız ağırlıklara bağlı alt ifadeleri önceden hesaplamak), **şekil çıkarımı** (bellek planlaması için), işlem füzyonu.

## 1 · Elle

Batch büyütmek verimi artırır ama tek isteğin gecikmesini de artırır (5 ms → 20 ms). Sunucular bu yüzden "dinamik batching" yapar: kısa bir süre bekleyip gelen istekleri birleştirir.

İzlenen graf yalnız $2x$ dalını içeriyor: negatif girdide de $2x$ hesaplanıyor.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**10 milyon parametreli model**

- `s32`: fp32 dosya boyutu (bayt)
- `s8`: int8 ağırlıklarla (bayt, ölçekleri yok say)

**Batch 1: 5 ms, batch 8: 20 ms**

- `t1`: Batch 1 verimi (örnek/s)
- `t8`: Batch 8 verimi (örnek/s)

**$f(x)=2x$ eğer $\sum x>0$, değilse $-x$. Kayıt tabanlı izleyici $x=(1,2)$ ile izliyor**

- `baked`: Kaydedilen graf $x=(-1,-1)$ ile çalışınca çıktının toplamı
- `right`: Doğru $\sum f(x)$

In [ ]:
cevaplar = dict(
    s32=None,
    s8=None,
    t1=None,
    t8=None,
    baked=None,
    right=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"s32":[40000000,0],"s8":[10000000,0],"t1":[200,0],"t8":[400,0],"baked":[-4,0],"right":[2,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python.

- `to_json(layers)` / `from_json(s)` — `layers`: `{"op": "linear", "W": …, "b": […]}` ya da `{"op": "relu"}` sözlükleri; `json` modülüyle.
- `run(layers, x)` — `x` liste; linear: $Wx+b$.
- Kayıt tabanlı izleyici: `Rec(values, graph)` — `*` (sayıyla), unary `-` işlemleri `graph`'a `("mul", c)` / `("neg",)` ekler ve yeni `Rec` döndürür; `sum()` **somut bir sayı** döndürür (kayıt yok).
- `trace(fn, x)` → graf listesi; `replay(graph, x)` → liste.

In [ ]:
%%writefile stage_b.py
import json


def to_json(layers):
    ...


def from_json(s):
    ...


def run(layers, x):
    ...


class Rec:
    def __init__(self, values, graph):
        self.values, self.graph = list(values), graph

    def __mul__(self, c):
        ...

    __rmul__ = __mul__

    def __neg__(self):
        ...

    def sum(self):
        ...


def trace(fn, x):
    ...


def replay(graph, x):
    ...


f = lambda x: x * 2 if x.sum() > 0 else -x
g = trace(f, [1.0, 2.0])
print(g, replay(g, [-1.0, -1.0]))   # [('mul', 2)] [-2, -2]  (yanlış dal donmuş)

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy ile ONNX benzeri mini çalışma zamanı. Düğüm: `{"op", "inputs", "output", …}`; işlemler `Gemm` ($xW^\top+b$), `Relu`, `Add`, `Mul`, `Transpose`, `Softmax` (son eksen). `inits`: sabit tensörler (ağırlıklar).

- `run_graph(nodes, inits, feeds)` → tüm değerler sözlüğü.
- `fold_constants(nodes, inits)` → `(yeni_düğümler, yeni_inits)`: tüm girdileri `inits`'te olan düğümleri hesapla, çıktılarını `inits`'e taşı, düğümü sil.
- `infer_shapes(nodes, inits, input_shapes)` → `{ad: şekil}`; şekiller tuple, sembolik boyutlar dize (`"N"`). Yayınlama (`Add`, `Mul`) için: eşitse o, biri 1 ise diğeri.

In [ ]:
%%writefile stage_c.py
import numpy as np


def run_graph(nodes, inits, feeds):
    ...


def fold_constants(nodes, inits):
    ...


def infer_shapes(nodes, inits, input_shapes):
    ...


rng = np.random.default_rng(0)
inits = {"W1": rng.normal(size=(8, 4)), "b1": np.zeros(8), "two": np.array(2.0), "W2": rng.normal(size=(8, 3)), "b2": np.zeros(3)}
nodes = [
    {"op": "Gemm", "inputs": ["x", "W1", "b1"], "output": "h"},
    {"op": "Relu", "inputs": ["h"], "output": "r"},
    {"op": "Mul", "inputs": ["W2", "two"], "output": "W2s"},
    {"op": "Transpose", "inputs": ["W2s"], "output": "W2t"},
    {"op": "Gemm", "inputs": ["r", "W2t", "b2"], "output": "y"},
    {"op": "Softmax", "inputs": ["y"], "output": "p"},
]
print(run_graph(nodes, inits, {"x": rng.normal(size=(5, 4))})["p"].sum(1))
n2, i2 = fold_constants(nodes, inits)
print([n["op"] for n in n2])
print(infer_shapes(nodes, inits, {"x": ("N", 4)}))

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

- `export_dynamic(model, example)` → `ExportedProgram`; batch boyutu dinamik: `dynamic_shapes={"x": {0: Dim("batch", min=1, max=1024)}}` (anahtar, `forward`'un argüman adı).
- `op_names(ep)` → `call_function` düğümlerinin hedeflerinin dize hâlleri.
- `save_load(ep)` → `torch.export.save` ile `io.BytesIO`'ya yaz, `torch.export.load` ile geri oku.
- `CondModel` — `forward(x)`: `torch.cond(x.sum() > 0, lambda x: x * 2, lambda x: -x, (x,))`.
- `BadModel` — aynı mantık, Python `if` ile (kontrol export'un hata verdiğini doğrular).

In [ ]:
%%writefile stage_d.py
import io
import torch
from torch.export import export, Dim


def export_dynamic(model, example):
    ...


def op_names(ep):
    ...


def save_load(ep):
    ...


class CondModel(torch.nn.Module):
    def forward(self, x):
        ...


class BadModel(torch.nn.Module):
    def forward(self, x):
        if x.sum() > 0:
            return x * 2
        return -x


class MLP(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.net = torch.nn.Sequential(torch.nn.Linear(16, 32), torch.nn.ReLU(), torch.nn.Linear(32, 4))

    def forward(self, x):
        return self.net(x)


ep = export_dynamic(MLP().eval(), torch.randn(4, 16))
print(op_names(ep), ep.module()(torch.randn(7, 16)).shape)

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

AOTInductor: ExportedProgram'ı derlenmiş bir pakete çevir ve Python'dan yükle.

- `compile_package(ep, path)` → paket yolu (`torch._inductor.aoti_compile_and_package(ep, package_path=path)`).
- `load_package(path)` → çağrılabilir (`torch._inductor.aoti_load_package`).

İlk derleme CPU'da ~20–60 sn sürer (C++ derleyicisi gerekir). Kontrol farklı batch boyutlarında çıktıyı ve batch 1 gecikmesini eager ile karşılaştırır.

In [ ]:
%%writefile stage_e.py
import logging
import os
import tempfile
import torch
from torch.export import export, Dim

torch._logging.set_logs(inductor=logging.ERROR, dynamo=logging.ERROR)


class MLP(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.net = torch.nn.Sequential(torch.nn.Linear(64, 256), torch.nn.GELU(), torch.nn.Linear(256, 256), torch.nn.GELU(), torch.nn.Linear(256, 10))

    def forward(self, x):
        return self.net(x)


def compile_package(ep, path):
    ...


def load_package(path):
    ...


torch.manual_seed(0)
m = MLP().eval()
ep = export(m, (torch.randn(4, 64),), dynamic_shapes={"x": {0: Dim("batch", min=1, max=1024)}})
path = compile_package(ep, os.path.join(tempfile.mkdtemp(), "mlp.pt2"))
run = load_package(path)
print((run(torch.randn(9, 64)) - m(torch.randn(9, 64))).shape)

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Kaynaklar

- [PyTorch Docs — torch.export](https://docs.pytorch.org/docs/stable/export.html) — BSD-3-Clause
- [PyTorch Docs — AOTInductor](https://docs.pytorch.org/docs/stable/torch.compiler_aot_inductor.html) — BSD-3-Clause
- [PyTorch Docs — torch.cond](https://docs.pytorch.org/docs/stable/cond.html) — BSD-3-Clause
- [ONNX — Intermediate Representation spec](https://onnx.ai/onnx/repo-docs/IR.html) — Apache-2.0
- [ONNX Runtime Web](https://onnxruntime.ai/docs/tutorials/web/) — MIT